In [ ]:
import pandas as pd


orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
tables = {
    'orders': orders, 'customers': customers, 'order_items': order_items,
    'payments': payments, 'reviews': reviews, 'products': products,
    'sellers': sellers, 'geolocation': geolocation, 'category_translation': category_translation
}

In [2]:
# orders table
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# reviews table
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# order_items table
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [4]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
status_counts = orders['order_status'].value_counts()
status_percentages = orders['order_status'].value_counts(normalize=True) * 100

print(status_counts)
print(status_percentages)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
order_status
delivered      97.020344
shipped         1.113223
canceled        0.628513
unavailable     0.612423
invoiced        0.315765
processing      0.302692
created         0.005028
approved        0.002011
Name: proportion, dtype: float64


### Order status filtering
- Total orders: 99,441
- Delivered: 96,478 (97.0%)
- Non-delivered (canceled/unavailable/shipped/etc.): 2,963 (3.0%)

Decision: `orders_delivered` (delivered-only) will be used for all revenue and
 delivery-time analysis going forward. The 3.0% non-delivered rate is reported 
separately as a fulfillment-failure metric, not dropped silently.

In [5]:
orders_delivered['in_analysis_window'] = (
    (orders_delivered['order_purchase_timestamp'] >= '2017-01-01') &
    (orders_delivered['order_purchase_timestamp'] <= '2018-08-31')
)

### Analysis window flag
Based on Phase 1 finding that the first/last months of data had very few orders
(incomplete collection), added `in_analysis_window` flag (True/False) marking
orders purchased between 2017-01-01 and 2018-08-31.
- In window: [96211] orders
- Outside window: [267] orders (excluded from trend analysis only, not deleted)

In [7]:
reviews = reviews.sort_values('review_answer_timestamp', ascending=False).drop_duplicates(subset='review_id', keep='first')

### Reviews deduplication
Found review_id duplicates not caught by full-row .duplicated() check 
(some reviews had multiple rows with different timestamps — likely 
 re-submitted/updated reviews). Deduplicated by keeping only the most 
recent version (latest review_answer_timestamp) per review_id.
- Rows before: [99224]
- Rows after: [98410]

In [9]:
geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

### Geolocation deduplication
Collapsed geolocation table from many rows per zip_code_prefix 
(261,831 duplicates found in Phase 1) down to one row per zip prefix, 
using mean lat/lng and first city/state per group. Result: clean lookup 
table for joining zip prefix → approximate coordinates.
- Rows before: [1000163]
- Rows after: [19015] (matches unique zip prefix count)

In [11]:
# 610 products are missing category_name (and correlated fields: name_lenght, 
# description_lenght, photos_qty), likely incomplete listings from sellers.
# Filling category with 'unknown' so these products remain visible in 
# category-based analysis rather than silently disappearing from groupby results.
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# name_lenght, description_lenght, photos_qty, and dimension fields left as NaN,
# these are used in aggregate/numeric analysis (e.g. averages), where pandas 
# handles NaN by excluding it automatically, so no fill needed here.

### Missing product data
610 products missing category_name/name_lenght/description_lenght/photos_qty 
(likely incomplete seller listings). 
Decision: filled product_category_name with 'unknown' to keep these products 
visible in category-based analysis. 
Left the length/count fields as NaN since numeric aggregations handle NaN 
automatically. 
2 products missing dimension fields (weight/length/height/width), 
left as NaN, no fill applied (no sensible default value exists for physical 
dimensions).

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

orders_delivered.to_csv('../data/processed/orders_clean.csv', index=False)
reviews.to_csv('../data/processed/reviews_clean.csv', index=False)
geolocation_clean.to_csv('../data/processed/geolocation_clean.csv', index=False)
products.to_csv('../data/processed/products_clean.csv', index=False)
customers.to_csv('../data/processed/customers_clean.csv', index=False)
order_items.to_csv('../data/processed/order_items_clean.csv', index=False)
payments.to_csv('../data/processed/payments_clean.csv', index=False)
sellers.to_csv('../data/processed/sellers_clean.csv', index=False)
category_translation.to_csv('../data/processed/category_translation_clean.csv', index=False)

### Exported cleaned data
Saved all cleaned tables to data/processed/ as CSVs, using orders_delivered 
(delivered-only, with in_analysis_window flag) 
instead of raw orders, deduplicated reviews and geolocation, 
and products with category nulls filled as 'unknown'. 
These files are the direct inputs for the Phase 2 analysis notebook.